In [1]:
# clone repo

import os

PROJECT_ROOT = "/content/Project_Generative_AI_for_Data_Augmentation"

if not os.path.exists(PROJECT_ROOT):
    !git clone https://github.com/Jorj91/Project_Generative_AI_for_Data_Augmentation.git {PROJECT_ROOT}

%cd {PROJECT_ROOT}

Cloning into '/content/Project_Generative_AI_for_Data_Augmentation'...
remote: Enumerating objects: 152, done.
remote: Counting objects: 100% (152/152), done.
remote: Compressing objects: 100% (129/129), done.
remote: Total 152 (delta 78), reused 66 (delta 20), pack-reused 0 (from 0)
Receiving objects: 100% (152/152), 7.50 MiB | 14.66 MiB/s, done.
Resolving deltas: 100% (78/78), done.
/content/Project_Generative_AI_for_Data_Augmentation


In [2]:
# =============================
# GLOBAL SILENT MODE
# =============================
import warnings

# Disable HF download progress bars
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"

# Silence transformers logging
os.environ["TRANSFORMERS_VERBOSITY"] = "error"

warnings.filterwarnings("ignore")

from transformers.utils import logging as transformers_logging
from huggingface_hub.utils import logging as hf_logging

transformers_logging.set_verbosity_error()
transformers_logging.disable_progress_bar()
hf_logging.set_verbosity_error()

In [3]:
# Setup

import torch
import logging
from torchvision.datasets import OxfordIIITPet
import numpy as np


PROJECT_ROOT = os.getcwd()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

logging.getLogger("transformers").setLevel(logging.ERROR)
logging.getLogger("huggingface_hub").setLevel(logging.ERROR)

In [4]:
PROJECT_ROOT # it must be /content/Project_Generative_AI_for_Data_Augmentation

'/content/Project_Generative_AI_for_Data_Augmentation'

In [5]:
# Dependency install
INSTALL_DEPS = True

if INSTALL_DEPS:
    !pip install -r requirements.txt -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 17.2 MB/s eta 0:00:00


In [6]:
# control flags
RUN_CAPTIONING = False
RUN_TEXT_VARIATION = True
RUN_IMAGE_GENERATION = False
RUN_TRAINING = False

In [7]:
# dataset loading

dataset_train = OxfordIIITPet(
    root=os.path.join(PROJECT_ROOT, "data", "raw"),
    split="trainval",
    download=True
)

dataset_test = OxfordIIITPet(
    root=os.path.join(PROJECT_ROOT, "data", "raw"),
    split="test",
    download=True
)

print("Train size:", len(dataset_train))
print("Test size:", len(dataset_test))

100%|██████████| 792M/792M [00:41<00:00, 19.3MB/s]
100%|██████████| 19.2M/19.2M [00:02<00:00, 9.33MB/s]


Train size: 3680
Test size: 3669


In [8]:
# load split

split_path = os.path.join(PROJECT_ROOT, "data", "splits", "train_small_indices.npy")

if os.path.exists(split_path):
    train_small_idx = np.load(split_path)
    print("Loaded 30% split:", len(train_small_idx))
else:
    print("Split not found. Run captioning notebook first.")

Loaded 30% split: 1104


In [9]:
# integrate 01_captioning (it takes 15 minutes to execute with A100 GPU on Colab)
if RUN_CAPTIONING:
    %run notebooks/01_captioning.ipynb

In [10]:
# integrate 02_text_variation
if RUN_TEXT_VARIATION:
    %run notebooks/02_text_variation.ipynb

Original: a pomeranian dog this dog is sitting on the bed
Generated: ['three-leggi', 'The dogs has taken shelter by taking photos, jumping into someone the dogs with some kind dogs of this puppy she got on that a very dandly friend by an owner so to the rest', 'Two pets lying naked. Some young child. He sit and stand outside his family, waiting when there should have started dog walk through the streets, after their food bowl is made of sugar dough (he']
Original: a pomeranian dog my dog is sitting on the bed
Generated: ['dog dog wearing we want his best day while not taking care her dogs nap with owner for pet care kashgarhian on line service app store by using my offer coupon free for ', 'dogs pear', 'A dog nexteer , pomelaid who can walk in city the day to come on stage. It was our favourite.Hea. It bes! We did so we named everything']
Original: a havanese dog is sitting on the tennis court
Generated: ['Some dogs playing down below from the sidewalk when I walk right below him becau

100%|██████████| 10/10 [00:43<00:00,  4.40s/it]


In [11]:

# save
# push

! git config --global user.email "fellinegiorgia@gmail.com"
! git config --global user.name "Jorj91"

!git remote remove origin

!git remote add origin https://MY_TOKEN@github.com/Jorj91/Project_Generative_AI_for_Data_Augmentation.git

! git push --set-upstream origin main

! git add .

! git commit -m "check 02 execution and main output"

! git push


Branch 'main' set up to track remote branch 'main' from 'origin'.
Everything up-to-date
[main 45ad4a3] check 02 execution and main output
 1 file changed, 34 insertions(+), 33 deletions(-)
Enumerating objects: 9, done.
Counting objects: 100% (9/9), done.
Delta compression using up to 12 threads
Compressing objects: 100% (5/5), done.
Writing objects: 100% (5/5), 1.05 KiB | 1.05 MiB/s, done.
Total 5 (delta 3), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (3/3), completed with 3 local objects.
To https://github.com/Jorj91/Project_Generative_AI_for_Data_Augmentation.git
   7acaf1c..45ad4a3  main -> main


In [ ]:
# import nbformat
# import os

# def hard_clean_notebook(path):
#     nb = nbformat.read(path, as_version=4)

#     if "widgets" in nb.metadata:
#         del nb.metadata["widgets"]

#     for cell in nb.cells:
#         if "widgets" in cell.get("metadata", {}):
#             del cell["metadata"]["widgets"]

#     nbformat.write(nb, path)
#     print(f"Cleaned: {os.path.basename(path)}")


# # 🔥 Walk entire project and clean every notebook
# for root, _, files in os.walk(PROJECT_ROOT):
#     for file in files:
#         if file.endswith(".ipynb"):
#             hard_clean_notebook(os.path.join(root, file))

# print("All notebooks in project HARD cleaned.")